In [1]:
import pandas as pd
import duckdb
import glob

In [41]:
path = "data/energy_charts/cross_border_electricity_trading/**/*.parquet"
file = glob.glob(path)[0]
dap = pd.read_parquet(file)

IndexError: list index out of range

In [45]:
read_parquet = duckdb.read_parquet(path)

In [67]:
read_parquet

┌─────────┬─────────┬────────────────┬─────────┬────────┬────────────┬─────────────┬────────┬────────┬────────┬─────────────┬────────┬──────────────┬─────────┬────────────────────┬────────────────┐
│ austria │ belgium │ czech_republic │ denmark │ france │ luxembourg │ netherlands │ norway │ poland │ sweden │ switzerland │  sum   │ unix_seconds │ country │    _dlt_load_id    │    _dlt_id     │
│ double  │ double  │     double     │ double  │ double │   double   │   double    │ double │ double │ double │   double    │ double │    int64     │ varchar │      varchar       │    varchar     │
├─────────┼─────────┼────────────────┼─────────┼────────┼────────────┼─────────────┼────────┼────────┼────────┼─────────────┼────────┼──────────────┼─────────┼────────────────────┼────────────────┤
│  -2.455 │   0.998 │         -0.357 │   2.029 │  2.846 │     -0.318 │       2.534 │    1.4 │ -0.898 │  0.615 │       1.652 │  8.045 │   1756764000 │ de      │ 1757265244.66022   │ aswYHVneopNl+A │
│  -2.395 

In [71]:
import duckdb

con = duckdb.connect()
path = "data/energy_charts/cross_border_electricity_trading/**/*.parquet"

# Duplicate values with counts
dupes = con.sql(f"""
with src as (
  select * from read_parquet('{path}')
),
tagged as (
  select
    *,
    row_number() over (partition by unix_seconds) as rn
  from src
)
select * 
from tagged
where rn = 1;
""")
print(dupes.to_df())  # or dupes.df() depending on your DuckDB version


     austria  belgium  czech_republic  denmark  france  luxembourg  \
0     -2.063    0.998          -0.195    2.152   2.862      -0.307   
1     -1.094    1.000          -0.047    2.194   2.224      -0.303   
2     -0.577    1.000          -0.917    1.111   3.871      -0.249   
3     -0.673    1.000          -0.885    0.886   3.871      -0.249   
4     -1.192    1.000          -1.096    0.510   2.997      -0.231   
..       ...      ...             ...      ...     ...         ...   
667   -1.868    1.000          -0.514   -0.040   0.962      -0.398   
668    0.086    1.000           0.368   -0.598   0.230      -0.453   
669   -1.006    0.966           0.068   -0.325   0.988      -0.390   
670   -1.717    0.986          -0.265   -0.114   1.055      -0.430   
671   -0.098   -0.000           0.206   -0.185   0.856      -0.465   

     netherlands  norway  poland  sweden  switzerland    sum  unix_seconds  \
0          2.542   1.400  -0.736   0.586        1.414  8.652    1756770300   
1  

Show duckDB data

In [2]:
con = duckdb.connect("dbt_project/energydata.duckdb")

In [3]:
con.execute("use analytics_staging")

In [49]:
print("Tables:", con.execute("show tables").fetchall())

Tables: [('stg_energy_charts__cbet',)]


In [17]:
schemas = con.execute("SELECT DISTINCT schema_name FROM information_schema.schemata").fetchall()
print("Schemas:", schemas)

Schemas: [('pg_catalog',), ('analytics_core',), ('main',), ('information_schema',), ('analytics_intermediate',), ('analytics_staging',)]


In [4]:
data = con.execute("select * from stg_energy_charts__cbet").df()
data

,austria,belgium,czech_republic,denmark,france,luxembourg,netherlands,norway,poland,sweden,switzerland,sum,unix_seconds,country,_dlt_load_id,_dlt_id
0,-2.455,0.998,-0.357,2.029,2.846,-0.318,2.534,1.4,-0.898,0.615,1.652,8.045,1756764000,de,1757265244.66022,aswYHVneopNl+A
1,-2.395,1.000,-0.350,1.990,2.846,-0.318,2.568,1.4,-0.890,0.615,1.652,8.118,1756764900,de,1757265244.66022,4Ta00C0P2Hgjsw
2,-2.103,0.963,-0.246,2.093,2.862,-0.307,2.541,1.4,-0.796,0.586,1.414,8.407,1756769400,de,1757265244.66022,OS2fsnxuNjzKEw
3,-1.626,0.999,-0.119,1.993,2.290,-0.297,2.278,1.4,-0.360,0.615,1.478,8.652,1756771200,de,1757265244.66022,XXkLD97DYrGs3g
4,0.876,0.991,-0.057,1.429,2.240,-0.444,2.147,1.4,-0.344,0.000,2.600,10.839,1756789200,de,1757265244.66022,d+/ENqeQXmg6GQ
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
667,-2.691,1.000,-0.606,0.278,0.919,-0.342,-0.859,0.0,-1.619,0.000,-0.328,-4.247,1757303100,de,1757265244.66022,c07vWniPK1pdtQ
668,-0.202,1.000,-0.043,-0.422,0.512,-0.446,-0.128,0.0,-0.903,0.000,2.268,1.637,1757307600,de,1757265244.66022,0qPidva7uHtX2A
669,-0.478,1.000,0.196,-0.905,1.141,-0.483,-0.500,0.0,0.314,0.000,-0.527,-0.243,1757317500,de,1757265244.66022,TZ7pxdM/vpb+wA
670,-0.286,1.000,0.135,-0.668,0.567,-0.407,-1.124,0.0,0.296,0.000,-1.525,-2.012,1757330100,de,1757265244.66022,DxeOuU3avVo1rg


In [5]:
con.close()